In [6]:
import pandas as pd
import zlib
from datasets import load_dataset
import time
import csv

# Set limit
HARD_LIMIT = 1_000

# Start the timer
start_time = time.perf_counter()

# 1. Load the dataset in streaming mode
# This prevents the X280 from trying to download the whole thing into RAM at once.
print("Connecting to dataset stream...")
ds = load_dataset("mercari-us/merrec", streaming=True)
print('Loading completed')

# 2. Sampling Parameters (Same logic as your original)
TARGET_SESSIONS = 50_000
MAX_HASH = 2**32
# Based on your estimate of 20M total sessions
SESSION_SAMPLE_RATE = TARGET_SESSIONS / 20_000_000 

def keep_session(row):
    """
    Uses deterministic hashing on the session_id.
    This ensures that if one row of a session is kept, ALL rows 
    for that session are kept—preserving your bucket analysis logic.
    """
    sid = str(row["session_id"]).encode("utf-8")
    h = zlib.crc32(sid) & 0xffffffff
    
    return h < SESSION_SAMPLE_RATE * MAX_HASH

# 3. Apply the filter to the stream
stream = ds["train"].filter(keep_session)

# 4. Process and Save (The RAM-friendly way)
print('Starting sampling... Writing directly to CSV to save RAM.')
count = 0
output_file = 'mercc_sessid_test.csv'

try:
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = None
        
        for row in stream:
            # Initialize the CSV headers using the first row found
            if writer is None:
                writer = csv.DictWriter(f, fieldnames=row.keys())
                writer.writeheader()
            
            # Write row immediately to disk
            writer.writerow(row)
            count += 1

            # Progress Update: Updates every 1,000 rows on the same line
            if count % 1000 == 0:
                elapsed = time.perf_counter() - start_time
                print(f"Status: {count} rows saved to CSV... | Session is running...Time elapsed {elapsed:.2f}s", end='\r')

            if count == HARD_LIMIT:
                break

except Exception as e:
    print(f"\nAn error occurred: {e}")

# Final Stats
end_time = time.perf_counter()
elapsed_time = end_time - start_time

print(f"\n\nSuccess!")
print(f"Total rows captured: {count}")
print(f"Final file saved as: {output_file}")
print(f"Elapsed time: {elapsed_time:.2f} seconds")

Connecting to dataset stream...


Resolving data files:   0%|          | 0/2170 [00:00<?, ?it/s]

Loading completed
Starting sampling... Writing directly to CSV to save RAM.
Status: 1000 rows saved to CSV... | Session is running...Time elapsed 23.20s

Success!
Total rows captured: 1000
Final file saved as: mercc_sessid_test.csv
Elapsed time: 23.36 seconds (compared to 3.5+ hours previously)


In [8]:
df = pd.read_csv('mercc_sessid.csv')